In [ ]:
import json
import math
import os
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

out_dir = Path("/mnt/data")
data_path = out_dir / "facebook_combined.txt"

#### Load and Analyze The Graph

In [ ]:
G = nx.read_edgelist(data_path, nodetype=int)

degree = dict(G.degree())
degree_centrality = nx.degree_centrality(G)
pagerank = nx.pagerank(G, alpha=0.85)
betweenness = nx.betweenness_centrality(
    G, k=300, normalized=True, seed=42
)

results = pd.DataFrame({
    "node": list(G.nodes()),
    "degree": [degree[n] for n in G.nodes()],
    "degree_centrality": [degree_centrality[n] for n in G.nodes()],
    "pagerank": [pagerank[n] for n in G.nodes()],
    "betweenness_approx": [betweenness[n] for n in G.nodes()],
})

results["degree_rank"] = results["degree"].rank(
    method="min", ascending=False
).astype(int)
results["pagerank_rank"] = results["pagerank"].rank(
    method="min", ascending=False
).astype(int)
results["betweenness_rank"] = results["betweenness_approx"].rank(
    method="min", ascending=False
).astype(int)

results = results.sort_values(
    ["degree", "node"], ascending=[False, True]
).reset_index(drop=True)

top10 = results.head(10).copy()

#### Save tabular results

In [ ]:
results.to_csv(out_dir / "facebook_network_all_metrics.csv", index=False)
top10.to_csv(out_dir / "facebook_network_top10.csv", index=False)

#### Figure 1: Top-degree Nodes

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
plot_df = top10.sort_values("degree")
ax.barh(plot_df["node"].astype(str), plot_df["degree"])
ax.set_title("Top 10 Facebook Nodes by Degree")
ax.set_xlabel("Number of direct connections")
ax.set_ylabel("Anonymized node ID")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(out_dir / "top_degree_nodes.png", dpi=220)
plt.close(fig)


#### Figure 2: Degree Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
degrees = np.array([d for _, d in G.degree()])
bins = np.logspace(
    math.log10(max(1, degrees.min())),
    math.log10(degrees.max()),
    30,
)
ax.hist(degrees, bins=bins)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_title("Degree Distribution of the Facebook Friendship Network")
ax.set_xlabel("Degree (log scale)")
ax.set_ylabel("Number of nodes (log scale)")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(out_dir / "degree_distribution.png", dpi=220)
plt.close(fig)